## Imports

In [ ]:
import os
import json
import random
import warnings
from pathlib import Path
import itertools

import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import balanced_accuracy_score
import torch

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 200)

## CFG

In [ ]:
class CFG:
    EXP_ID = "exp_20260416_046b_xgb_digit_orderedte_min_with_optuna_params"
    EXP_NAME = "xgb_digit_orderedte_min_with_optuna_params"
    SAVE_DIR = Path(f"/kaggle/working/{EXP_ID}")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"

    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"
    SEED = 2026
    N_FOLDS = 5
    NUM_CLASSES = 3

    XGB_PARAMS = {
        "max_depth": 4,
        "colsample_bytree": 0.8,
        "subsample": 0.8,
        "n_estimators": 512,
        "learning_rate": 0.1,
        "early_stopping_rounds": 100,
        "random_state": SEED,
        "n_jobs": -1,
        "enable_categorical": True,
        "alpha": 5,
        "reg_lambda": 5,
        "max_leaves": 30,
        "min_child_weight": 2,
        "tree_method": "hist",
        "max_bin": 10000,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    }

    best_params_payload = {}
    best_params_path = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/best_params.json"
    with open(best_params_path, mode="rt", encoding="utf-8") as f:
        best_params_payload = json.load(f)
    
    XGB_PARAMS.update(best_params_payload["best_params"])
    XGB_PARAMS["n_estimators"] = 2500

## utility

In [ ]:
def seed_everything(seed):
    np.random.seed(seed)
    random.seed(seed)

def balanced_acc_score_mc(y_true, y_pred):
    if len(y_pred.shape) == 2:
        y_pred = np.argmax(y_pred, axis=1)
    c = len(np.unique(y_true))
    acc = 0.0
    for i in range(c):
        acc += np.sum((y_true == i) & (y_pred == i)) / np.sum(y_true == i) / c
    return acc

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def safe_logit_like(proba, eps=1e-12):
    return np.log(np.clip(proba, eps, 1.0))

def apply_bias(proba, bias):
    """
    bias: iterable of len 3
    apply in log-prob space, then renormalize
    """
    logp = safe_logit_like(proba)
    adjusted = logp + np.array(bias, dtype=np.float32).reshape(1, -1)

    adjusted = adjusted - adjusted.max(axis=1, keepdims=True)
    expv = np.exp(adjusted)
    out = expv / expv.sum(axis=1, keepdims=True)
    return out.astype(np.float32)

def score_bias(proba, y_true, bias):
    adj = apply_bias(proba, bias)
    pred = np.argmax(adj, axis=1)
    return balanced_accuracy_score(y_true, pred)

def run_grid_search(proba, y_true, bias_ranges):
    best_bias = None
    best_score = -1.0

    for b0, b1, b2 in itertools.product(*bias_ranges):
        bias = (b0, b1, b2)
        sc = score_bias(proba, y_true, bias)
        if sc > best_score:
            best_score = sc
            best_bias = bias

    return best_bias, float(best_score)


seed_everything(CFG.SEED)

## load data

In [ ]:
drop_cols = [CFG.ID_COL]

train = pd.read_csv(CFG.TRAIN_PATH).drop(drop_cols, axis=1)
test = pd.read_csv(CFG.TEST_PATH).drop(drop_cols, axis=1)

print(f"train.shape={train.shape}, test.shape={test.shape}")

target2idx = {v: i for i, v in enumerate(train[CFG.TARGET_COL].unique())}
idx2target = {v: k for k, v in target2idx.items()}

train[CFG.TARGET_COL] = train[CFG.TARGET_COL].map(target2idx)

CATS = [c for c in test.columns if train[c].dtype == object]
NUMS = [c for c in test.columns if c not in CATS]

print("CATS:", CATS)
print("NUMS:", NUMS)
display(train.head())

## digit FE

In [ ]:
M = train[NUMS].max()

def add_digit_features(df):
    df = df.copy()

    for c in NUMS:
        for k in range(-4, 4):
            df[f"{c}_digit{k}"] = (df[c] // (10 ** k) % 10).astype("int8")

        if M[c] < 10:
            df[c] = df[c].round(3)
        elif M[c] < 100:
            df[c] = df[c].round(2)
        else:
            df[c] = df[c].round(1)

    return df

train = add_digit_features(train)
test = add_digit_features(test)

DROP = [c for c in test.columns if test[c].nunique() == 1]
print("DROP:", DROP)

train.drop(DROP, axis=1, inplace=True)
test.drop(DROP, axis=1, inplace=True)

## category remap

In [ ]:
CATEGORY = CATS + [c for c in test.columns if "digit" in c]

for c in CATEGORY:
    freq = train[c].value_counts()
    mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
    mapping_default = len(mapping)

    train[c] = train[c].map(lambda x: mapping.get(x, mapping_default))
    test[c] = test[c].map(lambda x: mapping.get(x, mapping_default))

FEATURES = CATEGORY + NUMS

print("n_CATEGORY =", len(CATEGORY))
print("n_FEATURES =", len(FEATURES))
display(train[FEATURES + [CFG.TARGET_COL]].head())

## class weight

In [ ]:
unique, counts = np.unique(train[CFG.TARGET_COL].values, return_counts=True)
count_dict = dict(zip(unique, counts))
avg_count = len(train) / len(unique)
weights_dict = {cls: avg_count / cnt for cls, cnt in count_dict.items()}
sample_weights = np.array([weights_dict[y] for y in train[CFG.TARGET_COL]])

print("weights_dict =", weights_dict)

## OrderedTE class

In [ ]:
class OrderedTE:
    def __init__(self, a=1):
        self.a = a

    def fit(self, train, category_cols=None, target_col="target"):
        if category_cols is None:
            category_cols = []

        self.train = train.copy()
        self.target_col = target_col
        self.category_cols = category_cols

        self.classes_ = sorted(train[target_col].unique())
        self.num_classes_ = len(self.classes_)
        self.global_prior_ = train[target_col].value_counts(normalize=True).sort_index().values

        for c in self.category_cols:
            stats_list = []

            for k, cls in enumerate(self.classes_):
                y_binary = (train[target_col] == cls).astype(int)

                df = train[[c]].copy()
                df["y"] = y_binary.values
                df["cnt"] = 1

                df["cum_cnt"] = df.groupby(c)["cnt"].cumsum() - df["cnt"]
                df["cum_sum"] = df.groupby(c)["y"].cumsum() - df["y"]

                smooth_prior = self.a * self.global_prior_[k]

                te_col = f"{c}_TE_cls{cls}"
                df[te_col] = (df["cum_sum"] + smooth_prior) / (df["cum_cnt"] + self.a)
                df.loc[df["cum_cnt"] == 0, te_col] = self.global_prior_[k]

                self.train[te_col] = df[te_col].values

                stats_df = df.groupby(c)["y"].agg(["count", "sum"]).reset_index()
                stats_df.columns = [c, f"{c}_count_cls{cls}", f"{c}_sum_cls{cls}"]
                stats_df[f"{c}_prior_cls{cls}"] = self.global_prior_[k]
                stats_list.append(stats_df)

            combined_stats = stats_list[0]
            for i in range(1, len(stats_list)):
                combined_stats = combined_stats.merge(stats_list[i], on=c, how="outer")
            setattr(self, f"{c}_stats", combined_stats)

        return self.train

    def transform(self, test):
        test = test.copy()

        for c in self.category_cols:
            stats_df = getattr(self, f"{c}_stats")
            test = test.merge(stats_df, on=c, how="left")

            for k, cls in enumerate(self.classes_):
                te_col = f"{c}_TE_cls{cls}"
                count_col = f"{c}_count_cls{cls}"
                sum_col = f"{c}_sum_cls{cls}"
                prior_col = f"{c}_prior_cls{cls}"

                if count_col in test.columns:
                    test[te_col] = (test[sum_col] + self.a * test[prior_col]) / (test[count_col] + self.a)
                    test[te_col] = test[te_col].fillna(test[prior_col])
                    test.drop([count_col, sum_col, prior_col], axis=1, inplace=True)
                else:
                    test[te_col] = self.global_prior_[k]

        return test

## CV main

In [ ]:
X = train.drop([CFG.TARGET_COL], axis=1)
y = train[CFG.TARGET_COL]
test_X = test.copy()

oof_raw = np.zeros((len(y), CFG.NUM_CLASSES))
pred_raw = np.zeros((len(test_X), CFG.NUM_CLASSES))

kf = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=42)
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    print(f"\nFold {fold}/{CFG.N_FOLDS}")

    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()
    train_weights = sample_weights[train_idx]

    te = OrderedTE(a=1)
    full_df = pd.concat((X_train, y_train), axis=1)
    full_df["weight"] = train_weights

    # 共有コードでは4回増殖しているが、今回は最小再現なので1回だけ
    te_train = te.fit(full_df.sample(frac=1, random_state=42), category_cols=FEATURES, target_col=CFG.TARGET_COL)

    X_train = te_train.drop([CFG.TARGET_COL, "weight"], axis=1)
    y_train = te_train[CFG.TARGET_COL]
    train_weights = te_train["weight"].values

    X_val = te.transform(X_val)
    X_test = te.transform(test_X)

    # 元カテゴリは落とす
    X_train.drop(CATS, axis=1, inplace=True)
    X_val.drop(CATS, axis=1, inplace=True)
    X_test.drop(CATS, axis=1, inplace=True)

    params = CFG.XGB_PARAMS.copy()
    if not torch.cuda.is_available():
        params["device"] = "cpu"

    model = XGBClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        sample_weight=train_weights,
        verbose=100,
    )

    y_pred = model.predict_proba(X_val)
    oof_raw[val_idx] = y_pred
    pred_raw += model.predict_proba(X_test) / CFG.N_FOLDS

    fold_ba = balanced_acc_score_mc(y_val.values, y_pred)
    fold_scores.append(float(fold_ba))
    print(f"fold_ba = {fold_ba:.6f}")

## raw CV

In [ ]:
raw_cv_ba = balanced_acc_score_mc(y.values, oof_raw)
print(f"raw_cv_ba = {raw_cv_ba:.6f}")

## bias search

In [ ]:
# ---------------------------------------------------------
# Stage 1: coarse
# ---------------------------------------------------------
coarse_vals = np.arange(-1.0, 1.01, 0.1)
best_bias_1, best_score_1 = run_grid_search(
    oof_raw, y,
    [coarse_vals, coarse_vals, coarse_vals]
)
print("stage1 best_bias:", best_bias_1, "score:", best_score_1)

In [ ]:
# ---------------------------------------------------------
# Stage 2: local refine
# ---------------------------------------------------------
b0, b1, b2 = best_bias_1
local_vals_0 = np.arange(b0 - 0.10, b0 + 0.1001, 0.02)
local_vals_1 = np.arange(b1 - 0.10, b1 + 0.1001, 0.02)
local_vals_2 = np.arange(b2 - 0.10, b2 + 0.1001, 0.02)

best_bias_2, best_score_2 = run_grid_search(
    oof_raw, y,
    [local_vals_0, local_vals_1, local_vals_2]
)
print("stage2 best_bias:", best_bias_2, "score:", best_score_2)

In [ ]:
# ---------------------------------------------------------
# Stage 3: ultra-local refine
# ---------------------------------------------------------
b0, b1, b2 = best_bias_2
ultra_vals_0 = np.arange(b0 - 0.02, b0 + 0.0201, 0.005)
ultra_vals_1 = np.arange(b1 - 0.02, b1 + 0.0201, 0.005)
ultra_vals_2 = np.arange(b2 - 0.02, b2 + 0.0201, 0.005)

best_bias_3, best_score_3 = run_grid_search(
    oof_raw, y,
    [ultra_vals_0, ultra_vals_1, ultra_vals_2]
)
print("stage3 best_bias:", best_bias_3, "score:", best_score_3)

final_bias = best_bias_3
final_cv = best_score_3

oof_biased = apply_bias(oof_raw, final_bias)
pred_biased = apply_bias(pred_raw, final_bias)

np.save(CFG.SAVE_DIR / "oof_xgb_digit_orderedte_min_optuna_biased_refined.npy", oof_biased)
np.save(CFG.SAVE_DIR / "pred_xgb_digit_orderedte_min_optuna_biased_refined.npy", pred_biased)
np.save(CFG.SAVE_DIR / "best_class_bias_refined.npy", np.array(final_bias, dtype=np.float32))

final_test_preds = np.argmax(pred_biased, axis=1)

submission = pd.read_csv(
    "/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv")
submission[CFG.TARGET_COL] = pd.Series(final_test_preds).map(idx2target)
submission.to_csv(CFG.SAVE_DIR / f"submission_{CFG.EXP_NAME}.csv", index=False)

cv_result = {
    "exp_id": CFG.EXP_ID,
    "base_parent": "exp_20260409_024_xgb_digit_orderedte_min",
    "raw_cv": float(raw_cv_ba),
    "refined_biased_cv": float(final_cv),
    "best_bias": list(map(float, final_bias)),
    "search": {
        "stage1": {"range": [-1.0, 1.0], "step": 0.1},
        "stage2": {"center": list(map(float, best_bias_1)), "width": 0.10, "step": 0.02},
        "stage3": {"center": list(map(float, best_bias_2)), "width": 0.02, "step": 0.005},
    }
}
save_json(cv_result, f"{CFG.SAVE_DIR}/cv_result.json")

print("final_bias:", final_bias)
print("final_cv:", final_cv)
print("saved to:", CFG.SAVE_DIR)